# Embedding Baseline Comparison

This notebook compares embedding-based approaches (SBERT + traditional classifiers) against transformer-based approaches.

**Approaches Compared:**
1. **SBERT Embeddings** + LogisticRegression
2. **SBERT Embeddings** + Linear SVM
3. **Transformer Classifier** (BERT-base + fine-tuning)

This provides insights into the tradeoff between embedding-based (simpler, faster) vs fine-tuned transformer (more complex, potentially higher accuracy) approaches.

In [ ]:
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")

print(f"✓ All imports successful")

## 1. Setup

In [ ]:
# Find repo root
def find_repo_root():
    current = Path.cwd()
    while current != current.parent:
        if (current / "src").exists() and (current / "README.md").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
DATA_DIR = SRC_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs" / "experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_ROOT))

# Global settings
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def set_seed(seed: int = SEED):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"✓ Device: {DEVICE}")
print(f"✓ Seed: {SEED}")
print(f"✓ Repo Root: {REPO_ROOT}")

## 2. Data Loading

In [ ]:
def load_jsonl(path: Path) -> List[dict]:
    """Load JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def label_to_int(label: str) -> int:
    """Convert label to int (fact=0, opinion=1)."""
    return 1 if label.lower() == "opinion" else 0

# Load data
print("📥 Loading data...")
train_data = load_jsonl(DATA_DIR / "news_train.jsonl")
test_data = load_jsonl(DATA_DIR / "news_test.jsonl")

train_texts = [d["text"] for d in train_data]
train_labels = np.array([label_to_int(d["label"]) for d in train_data])

test_texts = [d["text"] for d in test_data]
test_labels = np.array([label_to_int(d["label"]) for d in test_data])

print(f"✓ Train: {len(train_texts)} docs")
print(f"✓ Test: {len(test_texts)} docs")

## 3. SBERT Embedding-Based Approaches

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    SBERT_AVAILABLE = False
    print(f"⚠️  sentence-transformers not installed. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True

print(f"✓ SBERT available: {SBERT_AVAILABLE}")

In [ ]:
def get_embeddings_sbert(texts: List[str], model_name: str = "all-MiniLM-L6-v2") -> np.ndarray:
    """
    Get sentence embeddings using SBERT.
    Uses a smaller, faster model by default.
    """
    print(f"  Loading SBERT model: {model_name}")
    model = SentenceTransformer(model_name)
    print(f"  Encoding {len(texts)} documents...")
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
    print(f"  Embedding shape: {embeddings.shape}")
    return embeddings

print("✓ SBERT embedding function defined")

In [ ]:
print(f"\n{'='*70}")
print(f"EMBEDDING-BASED APPROACHES")
print(f"{'='*70}")

embedding_results = {}

# Get SBERT embeddings (using a smaller model for speed)
print(f"\n[1/2] Generating SBERT embeddings...")
sbert_model_name = "all-MiniLM-L6-v2"
try:
    train_embeddings = get_embeddings_sbert(train_texts, sbert_model_name)
    test_embeddings = get_embeddings_sbert(test_texts, sbert_model_name)
    print(f"✓ Embeddings generated successfully")
except Exception as e:
    print(f"✗ Error generating embeddings: {e}")
    train_embeddings = None
    test_embeddings = None

In [ ]:
if train_embeddings is not None:
    # ===== Logistic Regression =====
    print(f"\n[2/2] Training classifiers on SBERT embeddings...\n")
    
    print(f"  Training Logistic Regression...")
    lr_model = LogisticRegression(random_state=SEED, max_iter=1000)
    lr_model.fit(train_embeddings, train_labels)
    
    lr_train_preds = lr_model.predict(train_embeddings)
    lr_test_preds = lr_model.predict(test_embeddings)
    
    embedding_results["SBERT + LogisticRegression"] = {
        "model_type": "embedding",
        "train_f1_macro": f1_score(train_labels, lr_train_preds, average="macro"),
        "train_f1_weighted": f1_score(train_labels, lr_train_preds, average="weighted"),
        "train_acc": accuracy_score(train_labels, lr_train_preds),
        "test_f1_macro": f1_score(test_labels, lr_test_preds, average="macro"),
        "test_f1_weighted": f1_score(test_labels, lr_test_preds, average="weighted"),
        "test_acc": accuracy_score(test_labels, lr_test_preds),
        "cm": confusion_matrix(test_labels, lr_test_preds),
    }
    
    print(f"  ✓ Train F1: {embedding_results['SBERT + LogisticRegression']['train_f1_macro']:.4f} | Test F1: {embedding_results['SBERT + LogisticRegression']['test_f1_macro']:.4f}")
    
    # ===== Linear SVM =====
    print(f"  Training Linear SVM...")
    svm_model = svm.SVC(kernel="linear", random_state=SEED)
    svm_model.fit(train_embeddings, train_labels)
    
    svm_train_preds = svm_model.predict(train_embeddings)
    svm_test_preds = svm_model.predict(test_embeddings)
    
    embedding_results["SBERT + Linear SVM"] = {
        "model_type": "embedding",
        "train_f1_macro": f1_score(train_labels, svm_train_preds, average="macro"),
        "train_f1_weighted": f1_score(train_labels, svm_train_preds, average="weighted"),
        "train_acc": accuracy_score(train_labels, svm_train_preds),
        "test_f1_macro": f1_score(test_labels, svm_test_preds, average="macro"),
        "test_f1_weighted": f1_score(test_labels, svm_test_preds, average="weighted"),
        "test_acc": accuracy_score(test_labels, svm_test_preds),
        "cm": confusion_matrix(test_labels, svm_test_preds),
    }
    
    print(f"  ✓ Train F1: {embedding_results['SBERT + Linear SVM']['train_f1_macro']:.4f} | Test F1: {embedding_results['SBERT + Linear SVM']['test_f1_macro']:.4f}")
    
else:
    print(f"⚠️  Skipping embedding-based approaches due to errors")

## 4. Fine-tuned Transformer Classifier

In [ ]:
class DocumentDataset(Dataset):
    """PyTorch Dataset for document classification."""
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict:
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }

print("✓ DocumentDataset defined")

In [ ]:
def train_transformer(model, train_loader, device, epochs: int = 2, lr: float = 2e-5):
    """Fine-tune a transformer model."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)
    criterion = torch.nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
    
    return total_loss / (len(train_loader) * epochs)

def evaluate_transformer(model, loader, device):
    """Evaluate a transformer model."""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
    
    return np.array(all_preds), np.array(all_labels)

print("✓ Transformer training/evaluation functions defined")

In [ ]:
print(f"\n{'='*70}")
print(f"FINE-TUNED TRANSFORMER CLASSIFIER")
print(f"{'='*70}")

model_name = "bert-base-uncased"
print(f"\nModel: {model_name}")

try:
    # Initialize tokenizer and model
    print(f"Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print(f"Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(DEVICE)
    
    # Create datasets
    print(f"Creating datasets...")
    train_dataset = DocumentDataset(train_texts, train_labels, tokenizer, max_length=512)
    test_dataset = DocumentDataset(test_texts, test_labels, tokenizer, max_length=512)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=4)
    
    # Train
    print(f"\nTraining for 2 epochs...")
    train_transformer(model, train_loader, DEVICE, epochs=2, lr=2e-5)
    
    # Evaluate
    print(f"\nEvaluating...")
    train_preds, train_preds_labels = evaluate_transformer(model, train_loader, DEVICE)
    test_preds, test_preds_labels = evaluate_transformer(model, test_loader, DEVICE)
    
    embedding_results["BERT-base Fine-tuned"] = {
        "model_type": "transformer",
        "train_f1_macro": f1_score(train_preds_labels, train_preds, average="macro"),
        "train_f1_weighted": f1_score(train_preds_labels, train_preds, average="weighted"),
        "train_acc": accuracy_score(train_preds_labels, train_preds),
        "test_f1_macro": f1_score(test_preds_labels, test_preds, average="macro"),
        "test_f1_weighted": f1_score(test_preds_labels, test_preds, average="weighted"),
        "test_acc": accuracy_score(test_preds_labels, test_preds),
        "cm": confusion_matrix(test_preds_labels, test_preds),
    }
    
    print(f"✓ Train F1: {embedding_results['BERT-base Fine-tuned']['train_f1_macro']:.4f} | Test F1: {embedding_results['BERT-base Fine-tuned']['test_f1_macro']:.4f}")
    
except Exception as e:
    print(f"✗ Error training transformer: {e}")
    import traceback
    traceback.print_exc()

## 5. Results Summary

In [ ]:
# Create comprehensive results table
results_rows = []
for method, metrics in embedding_results.items():
    results_rows.append({
        "Approach": method,
        "Type": metrics["model_type"],
        "Train F1 (macro)": f"{metrics['train_f1_macro']:.4f}",
        "Test F1 (macro)": f"{metrics['test_f1_macro']:.4f}",
        "Test F1 (weighted)": f"{metrics['test_f1_weighted']:.4f}",
        "Test Accuracy": f"{metrics['test_acc']:.4f}",
    })

results_df = pd.DataFrame(results_rows)

print(f"\n" + "="*90)
print(f"EMBEDDING vs TRANSFORMER COMPARISON")
print(f"="*90)
print(results_df.to_string(index=False))
print(f"="*90)

## 6. Detailed Analysis

In [ ]:
# Find best approach
if embedding_results:
    best_method = max(embedding_results.items(), key=lambda x: x[1]["test_f1_macro"])
    
    print(f"\n" + "="*70)
    print(f"BEST APPROACH: {best_method[0]}")
    print(f"="*70)
    print(f"Test F1 (macro): {best_method[1]['test_f1_macro']:.4f}")
    print(f"Test F1 (weighted): {best_method[1]['test_f1_weighted']:.4f}")
    print(f"Test Accuracy: {best_method[1]['test_acc']:.4f}")
    print(f"\nConfusion Matrix:")
    print(best_method[1]["cm"])
    print(f"\n  TN={best_method[1]['cm'][0,0]}, FP={best_method[1]['cm'][0,1]}")
    print(f"  FN={best_method[1]['cm'][1,0]}, TP={best_method[1]['cm'][1,1]}")

## 7. Key Insights

In [ ]:
print(f"\n" + "="*70)
print(f"KEY INSIGHTS")
print(f"="*70)

if embedding_results:
    embedding_methods = {k: v for k, v in embedding_results.items() if v["model_type"] == "embedding"}
    transformer_methods = {k: v for k, v in embedding_results.items() if v["model_type"] == "transformer"}
    
    if embedding_methods:
        best_emb = max(embedding_methods.items(), key=lambda x: x[1]["test_f1_macro"])
        print(f"\n✓ Best Embedding-based: {best_emb[0]}")
        print(f"  Test F1 (macro): {best_emb[1]['test_f1_macro']:.4f}")
    
    if transformer_methods:
        best_trans = max(transformer_methods.items(), key=lambda x: x[1]["test_f1_macro"])
        print(f"\n✓ Best Fine-tuned Transformer: {best_trans[0]}")
        print(f"  Test F1 (macro): {best_trans[1]['test_f1_macro']:.4f}")
    
    # Compare
    if embedding_methods and transformer_methods:
        emb_f1 = best_emb[1]["test_f1_macro"]
        trans_f1 = best_trans[1]["test_f1_macro"]
        diff = trans_f1 - emb_f1
        pct_diff = (diff / emb_f1) * 100 if emb_f1 > 0 else 0
        
        print(f"\n📊 Comparison:")
        if diff > 0:
            print(f"  → Transformer is better by {diff:.4f} F1 ({pct_diff:+.1f}%)")
        elif diff < 0:
            print(f"  → Embedding approach is better by {abs(diff):.4f} F1 ({pct_diff:+.1f}%)")
        else:
            print(f"  → Both approaches have similar performance")

print(f"\n✓ Analysis complete!")

## 8. Save Results

In [ ]:
# Save results to CSV
results_csv = OUTPUT_DIR / f"embedding_comparison_{TIMESTAMP}.csv"

export_data = []
for method, metrics in embedding_results.items():
    export_data.append({
        "Method": method,
        "Type": metrics["model_type"],
        "Train_F1_Macro": metrics["train_f1_macro"],
        "Train_F1_Weighted": metrics["train_f1_weighted"],
        "Train_Accuracy": metrics["train_acc"],
        "Test_F1_Macro": metrics["test_f1_macro"],
        "Test_F1_Weighted": metrics["test_f1_weighted"],
        "Test_Accuracy": metrics["test_acc"],
    })

export_df = pd.DataFrame(export_data)
export_df.to_csv(results_csv, index=False)
print(f"✓ Results saved to: {results_csv}")
print(f"\n📁 All results location: {OUTPUT_DIR}")

In [ ]:
print(f"\n" + "="*70)
print(f"SUMMARY")
print(f"="*70)
print(f"\nApproaches tested: {len(embedding_results)}")
print(f"Results saved: {results_csv.name}")
print(f"Timestamp: {TIMESTAMP}")
print(f"\n✓ Embedding vs Transformer comparison complete!")